In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, brier_score
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

In [3]:
#=== Load the data ===#
# Load
train_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/train.csv").drop("event_id", axis=1)
test_df  = pd.read_csv("WiDSWorldWide_GlobalDathon26/test.csv")

X = train_df.drop(["event", "time_to_hit_hours"], axis=1)
y_event = train_df["event"].astype(bool).to_numpy()
y_time  = train_df["time_to_hit_hours"].to_numpy()

y = np.array(list(zip(y_event, y_time)), dtype=[("event", bool), ("time", float)])
X_test = test_df.drop(["event_id"], axis=1)

times = np.array([12.0, 24.0, 48.0, 72.0])
times_eval = np.array([24.0, 48.0, 72.0])
weights = {24.0: 0.3, 48.0: 0.4, 72.0: 0.3}


In [4]:
def weighted_brier(y_train_fold, y_val_fold, surv_matrix, times_eval):
    _, bs = brier_score(y_train_fold, y_val_fold, surv_matrix, times_eval)
    return float(sum(weights[t] * s for t, s in zip(times_eval, bs)))

def hybrid_score(c_index, w_brier):
    return 0.3 * c_index + 0.7 * (1.0 - w_brier)

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_alpha_global = None
best_hybrid_global = -1

test_preds_folds = []

In [25]:


for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_event), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    #impute missing values. 
    pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
    )

    X_tr_s = pipe.fit_transform(X_tr)
    X_va_s = pipe.transform(X_va)
    X_te_s = pipe.transform(X_test)


    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_test)

    cox_path = CoxnetSurvivalAnalysis(
        l1_ratio=0.05,          # <<< small elastic-net helps stability (try 0.01–0.2)
        alpha_min_ratio=0.01,
        n_alphas=80
    )
    cox_path.fit(X_tr_s, y_tr)

    best_alpha_fold = None
    best_hybrid_fold = -1

    for a in cox_path.alphas_:
        # refit with baseline so we can score Brier properly for this alpha
        cox_tmp = CoxnetSurvivalAnalysis(alphas=[a], l1_ratio=0.05, fit_baseline_model=True)
        cox_tmp.fit(X_tr_s, y_tr)

        # C-index on validation
        risk = cox_tmp.predict(X_va_s)
        c_idx = concordance_index_censored(y_va["event"], y_va["time"], risk)[0]

        # Brier on validation @24/48/72 (drop times beyond fold max)
        max_t = y_va["time"].max()
        t_eval = times_eval[times_eval < max_t]
        if len(t_eval) == 0:
            continue

        surv_funcs = cox_tmp.predict_survival_function(X_va_s)
        surv_mat = np.array([np.interp(t_eval, fn.x, fn.y) for fn in surv_funcs])
        
        if not np.isfinite(surv_mat).all():
            continue # skip if any NaNs/Infs in survival predictions, temp fix will come back later to probably fix this

        w_bs = weighted_brier(y_tr, y_va, surv_mat, t_eval)
        h = hybrid_score(c_idx, w_bs)

        if h > best_hybrid_fold:
            best_hybrid_fold = h
            best_alpha_fold = a

    print(f"Fold {fold}: best_alpha={best_alpha_fold}, best_hybrid={best_hybrid_fold:.5f}")

    # track global best alpha across folds (simple approach: keep the best fold)
    if best_hybrid_fold > best_hybrid_global:
        best_hybrid_global = best_hybrid_fold
        best_alpha_global = best_alpha_fold

    # OPTIONAL: fold test preds (if you still want averaging)
    cox_best = CoxnetSurvivalAnalysis(alphas=[best_alpha_fold], l1_ratio=0.05, fit_baseline_model=True)
    cox_best.fit(X_tr_s, y_tr)

    
    
    surv_test = cox_best.predict_survival_function(X_te_s)
    probs = []
    for fn in surv_test:
        s = np.interp(times, fn.x, fn.y)
        p = 1 - s
        p = np.clip(p, 1e-6, 1 - 1e-6)   # <<< tiny clip only
        probs.append(p)
    test_preds_folds.append(np.array(probs))

print("Chosen best_alpha_global:", best_alpha_global)


/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: UserWarning: all coefficients are zero, consider decreasing alpha.
  cox_tmp.fit(X_tr_s, y_tr)


Fold 1: best_alpha=0.2155592877485528, best_hybrid=0.91187


/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: UserWarning: all coefficients are zero, consider decreasing alpha.
  cox_tmp.fit(X_tr_s, y_tr)


Fold 2: best_alpha=0.05329183531732352, best_hybrid=0.90079


/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: UserWarning: all coefficients are zero, consider decreasing alpha.
  cox_tmp.fit(X_tr_s, y_tr)
/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  cox_tmp.fit(X_tr_s, y_tr)


...

/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: UserWarning: all coefficients are zero, consider decreasing alpha.
  cox_tmp.fit(X_tr_s, y_tr)


Fold 4: best_alpha=0.05370170487720837, best_hybrid=0.92610


/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: UserWarning: all coefficients are zero, consider decreasing alpha.
  cox_tmp.fit(X_tr_s, y_tr)
/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  cox_tmp.fit(X_tr_s, y_tr)
/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  cox_tmp.fit(X_tr_s, y_tr)
/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  cox_tmp.fit(X_tr_s, y_tr)
/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: ConvergenceWarning: Optimization terminated e

Fold 5: best_alpha=0.07782637731366543, best_hybrid=0.87549
Chosen best_alpha_global: 0.05370170487720837


/var/folders/n5/rjbcg9dj5zv3ghm5byyll9700000gn/T/ipykernel_5470/4185349481.py:34: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=100000).
  cox_tmp.fit(X_tr_s, y_tr)
/Users/shahzadiaiman/Library/Python/3.9/lib/python/site-packages/sksurv/linear_model/coxph.py:81: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)


In [15]:
scaler = StandardScaler()
X_all_s = scaler.fit_transform(X)
X_test_s = scaler.transform(X_test)

In [26]:
final_model = CoxnetSurvivalAnalysis(alphas=[best_alpha_global], l1_ratio=0.05, fit_baseline_model=True)
final_model.fit(X_all_s, y)

CoxnetSurvivalAnalysis(alphas=[np.float64(0.05370170487720837)],
                       fit_baseline_model=True, l1_ratio=0.05)

In [27]:
surv_test = final_model.predict_survival_function(X_test_s)
final_probs = np.array([
    np.clip(1 - np.interp(times, fn.x, fn.y), 1e-6, 1 - 1e-6)
    for fn in surv_test
])

In [28]:
sub_new = pd.DataFrame({
    "event_id": test_df["event_id"],
    "prob_12h": final_probs[:, 0],
    "prob_24h": final_probs[:, 1],
    "prob_48h": final_probs[:, 2],
    "prob_72h": final_probs[:, 3],
})
sub_new.to_csv("sub_new.csv", index=False)
print("Saved sub_new.csv")

Saved sub_new.csv
